# Weather Agent — Gemini-Powered Multi-Step Weather Analysis

This notebook consolidates the original multi-file Weather Agent project. It uses Gemini for planning/tool selection and a local CSV dataset for grounded weather results.

**Audit changes:** fixed model configuration (uses `gemini-3.6-flash` from the original project), improved key retrieval, dataset upload/path selection, safer calculator limits, clearer date/data validation, and tool-call handling. The source ZIP did **not** contain the weather CSV, so you must upload `06_agent_weather_tool_data.csv` or set its path below.

No OpenAI API key or OpenAI SDK is used.

## 1. Install dependencies

In [ ]:
!pip -q install -U google-genai pandas python-dotenv

## 2. Load Gemini API key
The loader checks environment/`.env` first, then Google Colab Secrets (`GEMINI_API_KEY`). It never prints the key.

In [ ]:
import os
from pathlib import Path

# Optional .env support. If python-dotenv is available, load it.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def get_gemini_api_key():
    key = os.getenv("GEMINI_API_KEY")
    if key:
        return key.strip()
    # Google Colab Secret fallback
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return str(key).strip()
    except Exception:
        pass
    raise RuntimeError(
        "Gemini API key not found. Set GEMINI_API_KEY in a .env file/environment, "
        "or add GEMINI_API_KEY in Colab Secrets (key icon)."
    )

GEMINI_API_KEY = get_gemini_api_key()
print("Gemini API key loaded securely.")

## 3. Locate or upload the weather dataset

The original ZIP has no `data/` directory or CSV dataset. Upload the expected CSV below, or edit `DATA_PATH` to a file already in your runtime.

In [ ]:
DATA_PATH = "data/06_agent_weather_tool_data.csv"

if not Path(DATA_PATH).exists():
    try:
        from google.colab import files
        print("Upload the weather CSV file (expected columns: city, date, temperature_c, rainfall_mm, humidity_pct).")
        uploaded = files.upload()
        if uploaded:
            uploaded_name = next(iter(uploaded))
            DATA_PATH = uploaded_name
    except ImportError:
        print("Not running in Colab. Set DATA_PATH to your dataset CSV path.")

print("Dataset path:", DATA_PATH)

## 4. Imports and weather data loader

In [ ]:
import ast
import json
import math
import operator
import pandas as pd
from typing import Any, Dict, List, Optional, Union
from google import genai
from google.genai import types

class WeatherDataLoader:
    REQUIRED_COLUMNS = {"city", "date", "temperature_c", "rainfall_mm", "humidity_pct"}
    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self._df = self._load_and_validate_data()

    def _load_and_validate_data(self):
        if not os.path.isfile(self.csv_path):
            raise FileNotFoundError(f"Dataset not found: {self.csv_path}. Upload the CSV or set DATA_PATH.")
        df = pd.read_csv(self.csv_path)
        missing = self.REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
        df["city"] = df["city"].fillna("").astype(str).str.strip().str.title()
        parsed_dates = pd.to_datetime(df["date"], errors="coerce")
        df["date"] = parsed_dates.dt.strftime("%Y-%m-%d")
        for col in ["temperature_c", "rainfall_mm", "humidity_pct"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(subset=["city", "date", "temperature_c"])
        df = df[df["city"] != ""].copy()
        if df.empty:
            raise ValueError("Dataset contains no usable records after cleaning.")
        return df

    def get_city_weather(self, city: str, date: Optional[str] = None) -> List[Dict[str, Any]]:
        city_norm = str(city).strip().title()
        subset = self._df[self._df["city"] == city_norm]
        if date:
            date_norm = pd.to_datetime(date, errors="coerce")
            if pd.isna(date_norm):
                return []
            subset = subset[subset["date"] == date_norm.strftime("%Y-%m-%d")]
        return subset.to_dict(orient="records")

    def get_city_stats(self, city: str) -> Dict[str, Any]:
        city_norm = str(city).strip().title()
        subset = self._df[self._df["city"] == city_norm]
        if subset.empty:
            return {"found": False, "message": f"No records found for city '{city_norm}' in local dataset."}
        return {
            "found": True, "city": city_norm, "record_count": int(len(subset)),
            "average_temperature": round(float(subset["temperature_c"].mean()), 2),
            "maximum_temperature": round(float(subset["temperature_c"].max()), 2),
            "minimum_temperature": round(float(subset["temperature_c"].min()), 2),
            "total_rainfall": round(float(subset["rainfall_mm"].sum()), 2),
            "average_humidity": (round(float(subset["humidity_pct"].mean()), 2) if subset["humidity_pct"].notna().any() else None),
        }
    def list_available_cities(self):
        return sorted(self._df["city"].dropna().unique().tolist())

data_loader = WeatherDataLoader(DATA_PATH)
print(f"Loaded {len(data_loader._df)} records for cities: {', '.join(data_loader.list_available_cities())}")

## 5. Safe calculator and tool registry

In [ ]:
class SafeCalculator:
    OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
           ast.USub: operator.neg, ast.UAdd: operator.pos}
    @classmethod
    def evaluate(cls, expression: str):
        try:
            if len(expression) > 200:
                raise ValueError("Expression is too long.")
            node = ast.parse(expression.strip(), mode="eval").body
            value = cls._eval(node)
            if not math.isfinite(float(value)):
                raise ValueError("Result is not finite.")
            return round(value, 4)
        except ZeroDivisionError:
            return "Error: Division by zero."
        except Exception as exc:
            return f"Error: Invalid mathematical expression '{expression}'. ({exc})"
    @classmethod
    def _eval(cls, node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            if abs(node.value) > 1e12:
                raise ValueError("Numeric literal exceeds limit.")
            return node.value
        if isinstance(node, ast.BinOp):
            if type(node.op) not in cls.OPS or type(node.op) in (ast.USub, ast.UAdd):
                raise ValueError("Operator not allowed.")
            left, right = cls._eval(node.left), cls._eval(node.right)
            if isinstance(node.op, ast.Pow) and (abs(right) > 10 or abs(left) > 1e6):
                raise ValueError("Exponent operation exceeds safety limits.")
            result = cls.OPS[type(node.op)](left, right)
            if abs(result) > 1e100:
                raise ValueError("Result exceeds safety limit.")
            return result
        if isinstance(node, ast.UnaryOp) and type(node.op) in cls.OPS:
            return cls.OPS[type(node.op)](cls._eval(node.operand))
        raise ValueError(f"Unsupported expression element: {type(node).__name__}")

class ToolRegistry:
    def __init__(self, loader, api_key):
        self.loader, self.api_key = loader, api_key
        self._client = None
    def _get_client(self):
        if self._client is None:
            self._client = genai.Client(api_key=self.api_key)
        return self._client
    def weather_search_tool(self, city: str, date: Optional[str] = None):
        records = self.loader.get_city_weather(city, date)
        if not records:
            return {"status": "not_found", "message": f"No weather records found for {city}" + (f" on {date}" if date else ""),
                    "grounding_info": f"Available cities: {', '.join(self.loader.list_available_cities())}"}
        return {"status": "success", "count": len(records), "data": records}
    def weather_statistics_tool(self, city: str):
        stats = self.loader.get_city_stats(city)
        if not stats.get("found"):
            return {"status": "not_found", "message": stats.get("message", f"City {city} not found"),
                    "grounding_info": f"Available cities: {', '.join(self.loader.list_available_cities())}"}
        return {"status": "success", "statistics": stats}
    def calculator_tool(self, expression: str):
        result = SafeCalculator.evaluate(expression)
        if isinstance(result, str) and result.startswith("Error:"):
            return {"status": "error", "message": result}
        return {"status": "success", "expression": expression, "result": result}
    def text_summarizer_tool(self, content: str, focus: str = "general summary"):
        try:
            response = self._get_client().models.generate_content(
                model=MODEL_ID, contents=f"Summarize these weather results focusing on {focus}. Use only the supplied information:\n{content}")
            return {"status": "success", "focus": focus, "summary": (response.text or "").strip()}
        except Exception as exc:
            return {"status": "error", "message": f"Summary generation failed: {exc}"}

MODEL_ID = "gemini-3.6-flash"
print("Tools and calculator initialized.")

## 6. Gemini Weather Agent

In [ ]:
class WeatherAgent:
    def __init__(self, loader, api_key, model_id=MODEL_ID):
        self.loader = loader
        self.api_key = api_key
        self.model_id = model_id
        self.tools = ToolRegistry(loader, api_key)
        self.client = genai.Client(api_key=api_key)
        self.execution_trace = []
    def _system_instruction(self):
        cities = ", ".join(self.loader.list_available_cities())
        return (
            "You are a weather analysis agent. Ground weather data answers only in the local dataset. "
            f"Available cities: [{cities}]. If a city is absent, say data is unavailable and list available cities. "
            "Use weather_search_tool for records, weather_statistics_tool for aggregates, calculator_tool for ALL calculations, "
            "and text_summarizer_tool for summaries. For multi-part requests, perform tools in sequence. "
            "Do not invent weather data. Present units and explain calculations clearly."
        )
    def _dispatch(self, name, args):
        mapping = {"weather_search_tool": self.tools.weather_search_tool,
                  "weather_statistics_tool": self.tools.weather_statistics_tool,
                  "calculator_tool": self.tools.calculator_tool,
                  "text_summarizer_tool": self.tools.text_summarizer_tool}
        if name not in mapping:
            return {"status": "error", "message": f"Unknown tool: {name}"}
        try:
            return mapping[name](**args)
        except Exception as exc:
            return {"status": "error", "message": f"Tool execution error: {exc}"}
    def run(self, query, max_steps=8):
        self.execution_trace = []
        config = types.GenerateContentConfig(
            system_instruction=self._system_instruction(),
            tools=[self.tools.weather_search_tool, self.tools.weather_statistics_tool,
                   self.tools.calculator_tool, self.tools.text_summarizer_tool],
            temperature=0.1)
        chat = self.client.chats.create(model=self.model_id, config=config)
        response = chat.send_message(query)
        step = 0
        while getattr(response, "function_calls", None) and step < max_steps:
            calls = list(response.function_calls)
            function_responses = []
            for call in calls:
                if step >= max_steps:
                    break
                step += 1
                args = dict(call.args or {})
                output = self._dispatch(call.name, args)
                self.execution_trace.append({"step": step, "tool_selected": call.name, "inputs": args, "output": output})
                function_responses.append(types.Part.from_function_response(name=call.name, response={"result": output}))
            if not function_responses:
                break
            # Return all function responses together so multi-tool calls are handled correctly.
            response = chat.send_message(function_responses)
        text_out = getattr(response, "text", None) or "Query completed, but Gemini returned no text response."
        if step >= max_steps and getattr(response, "function_calls", None):
            text_out += "\n\nNote: tool-call limit reached; response may be incomplete."
        return {"query": query, "final_response": text_out, "trace": self.execution_trace}
    def print_execution_trace(self):
        print("\n" + "=" * 68 + "\nAGENT EXECUTION TRACE\n" + "=" * 68)
        if not self.execution_trace:
            print("No tools invoked.")
            return
        for item in self.execution_trace:
            print(f"\n[Step {item['step']}] {item['tool_selected']}\nArguments: {json.dumps(item['inputs'], ensure_ascii=False)}\nOutput: {json.dumps(item['output'], indent=2, ensure_ascii=False)}")
        print("=" * 68)

agent = WeatherAgent(data_loader, GEMINI_API_KEY)
print(f"Weather Agent ready. Model: {MODEL_ID}")

## 7. Test calculator and data layer (no LLM call)

In [ ]:
# Lightweight local checks before calling Gemini
assert SafeCalculator.evaluate("(10 + 5) / 3") == 5.0
assert SafeCalculator.evaluate("10 / 0").startswith("Error:")
print("Calculator checks passed.")
print("Available cities:", data_loader.list_available_cities())
print("Sample statistics:", data_loader.get_city_stats(data_loader.list_available_cities()[0]))

## 8. Ask the Weather Agent

In [ ]:
query = input("Weather question: ").strip()
if query:
    result = agent.run(query)
    agent.print_execution_trace()
    print("\nFINAL ANSWER\n", result["final_response"])

## 9. Interactive session (optional)

In [ ]:
# Run this cell to ask multiple questions in this notebook session.
# Type exit to stop.
while True:
    user_query = input("\nWeather query (or exit): ").strip()
    if user_query.lower() in {"exit", "quit"}:
        print("Session ended.")
        break
    if not user_query:
        continue
    try:
        answer = agent.run(user_query)
        agent.print_execution_trace()
        print("\nAgent:", answer["final_response"])
    except Exception as exc:
        print(f"Runtime error: {exc}")